In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=cm.get_cmap("Dark2").colors)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

# Put the repo's `src/` on sys.path so `quantproteomicssimbox` imports even without an editable
# install. This notebook lives in notebooks/, so the repo's `src/` is at "../src".
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from quantproteomicssimbox.methods import intensity_method, stoichiometry_method, QUANT_METHODS
from quantproteomicssimbox.sweep import run_sweep, records_to_rows

C:\Users\student\AppData\Local\Temp\ipykernel_7932\4030008377.py:8: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  plt.rcParams['axes.prop_cycle'] = plt.cycler(color=cm.get_cmap("Dark2").colors)


## Experiment Setup

In [2]:
# Sets the experiment parameters. 
# Constant values are passed to the sweep's base/fixed parameters.
# *List* parameters are passed to the sweep's grid parameters.
gridable_params = dict(
    # Experiment truth parameters
    n_proteins = 5,
    protein_length = 200,
    abundance = 250,
    n_groups = 2, # aka conditions
    n_subjects = 15, # aka bioreps
    repeat_units = 0, # peptide repeats in the sequence at different start loci
    digestion = "per_subject", # 'per_subject' or 'per_copy'
    miscleavage_model = "global", # 'global' or 'bernoulli'
    miscleavage_rate = 0.25, 

    # Observation model parameters
    var_subject = 1,
    var_site = 1, 
    var_species = 1,
    position_aware = True, # if False, the repeat_units will be aggregated despite representing different spans.
    detection_limit = 0,
    missingness = 0.25
)

# Sets the grid of rollup/quantification metrics. 
# Default imports a set of 7 common quant methods. See `quantproteomicssimbox.methods.QUANT_METHODS`. 
# Additional methods can be created with the imported 
# `intensity_method()` and `stoichiometry_methods()` functions.
quant_methods = QUANT_METHODS

# Group/condition analysis filter. 
# Only peptides in at least that many samples/bioreps of any group/condition are kept for analysis. 
# See `quantproteomicssimbox.rollups.intensity.roll_up()``
min_per_group = 1

# Pseudo-randomness seed
seed = 42

# Number of sweep replicates
n_replicates = 5

# Set up sweep's `base` and `grid` arguments
base = {}
grid = {}
for k,v in gridable_params.items():
    if isinstance(v, list):
        grid[k] = v
    else: 
        base[k] = v


## Experiment Run

In [3]:
results = run_sweep(grid=grid, 
                    methods=quant_methods, 
                    n_replicates = n_replicates,
                    base=base, 
                    min_per_group=min_per_group,
                    seed = seed)

## Experiment Analysis

In [4]:
results_for_pd = records_to_rows(results)
df = pd.DataFrame(results_for_pd)
df

,method,rmse_mean,rmse_se,n
0,int_mean,0.799673,0.091672,5
1,int_median,0.800678,0.089968,5
2,int_sum,2.726489,0.344715,5
3,stoich_fraction,0.896686,0.101520,5
4,stoich_logit,7.435754,0.552067,5
5,stoich_pep_mean,0.896686,0.101520,5
6,stoich_pep_median,0.896686,0.101520,5
